# FlashInfer-Bench Demo

**FlashInfer-Bench** is a benchmark suite and production workflow designed to build a virtuous cycle of self-improving AI systems. It enables AI agents and human experts to collaboratively optimize GPU kernels that power large language models.

- **Documentation**: [bench.flashinfer.ai/docs](https://bench.flashinfer.ai/docs/)
- **Leaderboard**: [bench.flashinfer.ai](https://bench.flashinfer.ai/)
- **Blog Post**: [flashinfer.ai/2025/10/21/flashinfer-bench](https://flashinfer.ai/2025/10/21/flashinfer-bench.html)
- **GitHub**: [github.com/flashinfer-ai/flashinfer-bench](https://github.com/flashinfer-ai/flashinfer-bench)


---
## 1. Setup & Installation


In [5]:
# pip install flashinfer-bench, or pip install -v .e . 
import flashinfer_bench as fib

print(f"FlashInfer-Bench version: {fib.__version__}")

FlashInfer-Bench version: 0.0.0.dev0


---
## 2. The FlashInfer Pipeline

FlashInfer-Bench creates a **virtuous cycle** where AI agents and human experts collaboratively optimize GPU kernels:

1. **Definition** → Kernel contract (axes, inputs, outputs, reference implementation)
2. **Solution** → Optimized implementations from AI agents (GPT-5, Claude, Gemini) or human experts
3. **Workload** → Real-world shapes captured from production
4. **Evaluation** → Correctness and performance measurements
5. **Leaderboard & Apply** → Best kernels are ranked and automatically substituted in production


### 2.1 Loading the FlashInfer-Trace Dataset

The **FlashInfer-Trace** dataset contains kernel definitions, solutions (from AI agents and humans), workloads, and evaluation traces.


In [3]:
# Load the FlashInfer-Trace dataset
TRACE_PATH = "/sgl-workspace/flashinfer-trace"

trace_set = fib.TraceSet.from_path(TRACE_PATH)

# Display summary statistics
print("="*60)
print("         FLASHINFER-TRACE DATASET SUMMARY")
print("="*60)
print(f"\n📁 Dataset path: {TRACE_PATH}")
print(f"\n📊 Statistics:")
print(f"   • Total Definitions: {len(trace_set.definitions)}")
print(f"   • Total Solutions:   {sum(len(s) for s in trace_set.solutions.values())}")
print(f"   • Total Workloads:   {sum(len(w) for w in trace_set.workloads.values())}")
print(f"   • Total Traces:      {sum(len(t) for t in trace_set.traces.values())}")

# Show trace summary
summary = trace_set.summary()
print(f"\n✅ Passed evaluations: {summary['passed']}")
print(f"❌ Failed evaluations: {summary['failed']}")
if summary['avg_latency_ms']:
    print(f"⏱️  Avg latency: {summary['avg_latency_ms']:.4f} ms")


         FLASHINFER-TRACE DATASET SUMMARY

📁 Dataset path: /sgl-workspace/flashinfer-trace

📊 Statistics:
   • Total Definitions: 35
   • Total Solutions:   314
   • Total Workloads:   663
   • Total Traces:      5908

✅ Passed evaluations: 3879
❌ Failed evaluations: 2029
⏱️  Avg latency: 11.2371 ms


### 2.2 Exploring Kernel Definitions

A **Definition** specifies the kernel's contract: axes (dimensions), inputs, outputs, and a reference implementation.


In [7]:
# List all available definitions by operation type
from collections import defaultdict
definitions_by_type = defaultdict(list)
for name, defn in trace_set.definitions.items():
    definitions_by_type[defn.op_type].append(name)

print("="*60)
print("         KERNEL DEFINITIONS BY OPERATION TYPE")
print("="*60)

for op_type, defs in sorted(definitions_by_type.items()):
    print(f"\n🔷 {op_type.upper()} ({len(defs)} definitions)")
    for d in sorted(defs):
        print(f"   └── {d}")


         KERNEL DEFINITIONS BY OPERATION TYPE

🔷 GEMM (8 definitions)
   └── gemm_n128_k2048
   └── gemm_n2048_k4096
   └── gemm_n256_k7168
   └── gemm_n28672_k4096
   └── gemm_n4096_k14336
   └── gemm_n4096_k4096
   └── gemm_n5120_k2048
   └── gemm_n6144_k4096

🔷 GQA_PAGED (4 definitions)
   └── gqa_paged_decode_h32_kv4_d128_ps1
   └── gqa_paged_decode_h32_kv8_d128_ps1
   └── gqa_paged_prefill_causal_h32_kv4_d128_ps1
   └── gqa_paged_prefill_causal_h32_kv8_d128_ps1

🔷 GQA_RAGGED (2 definitions)
   └── gqa_ragged_prefill_causal_h32_kv4_d128
   └── gqa_ragged_prefill_causal_h32_kv8_d128

🔷 MLA_PAGED (2 definitions)
   └── mla_paged_decode_h16_ckv512_kpe64_ps1
   └── mla_paged_prefill_causal_h16_ckv512_kpe64_ps1

🔷 MOE (1 definitions)
   └── moe_fp8_block_scale_ds_routing_topk8_ng8_kg4_e32_h7168_i2048

🔷 RMSNORM (9 definitions)
   └── fused_add_rmsnorm_h2048
   └── fused_add_rmsnorm_h4096
   └── fused_add_rmsnorm_h7168
   └── rmsnorm_h128
   └── rmsnorm_h1536
   └── rmsnorm_h2048
   └── 

In [8]:
# Examine a specific definition: fused_add_rmsnorm_h4096 (used in Llama-3.1-8B)
def_name = "fused_add_rmsnorm_h4096"
defn = trace_set.definitions[def_name]

print("="*60)
print(f"      DEFINITION: {def_name}")
print("="*60)

print(f"\n📝 Description: {defn.description}")
print(f"🏷️  Tags: {defn.tags}")
print(f"🔧 Op Type: {defn.op_type}")

print(f"\n📐 Axes (dimensions):")
for axis_name, axis_spec in defn.axes.items():
    if axis_spec.type == "const":
        print(f"   • {axis_name}: const = {axis_spec.value}")
    else:
        print(f"   • {axis_name}: variable (runtime)")

print(f"\n📥 Inputs:")
for inp_name, inp_spec in defn.inputs.items():
    print(f"   • {inp_name}: shape={inp_spec.shape}, dtype={inp_spec.dtype.value}")

print(f"\n📤 Outputs:")
for out_name, out_spec in defn.outputs.items():
    print(f"   • {out_name}: shape={out_spec.shape}, dtype={out_spec.dtype.value}")

print(f"\n📜 Reference Implementation:")
print("-"*60)
print(defn.reference)
print("-"*60)


      DEFINITION: fused_add_rmsnorm_h4096

📝 Description: Fused Add + RMSNorm with hidden_size=4096 for Llama-3.1-8B. Epsilon is fixed at 1e-5.
🏷️  Tags: ['status:verified', 'model:llama-3.1-8b', 'fused']
🔧 Op Type: rmsnorm

📐 Axes (dimensions):
   • batch_size: variable (runtime)
   • hidden_size: const = 4096

📥 Inputs:
   • hidden_states: shape=['batch_size', 'hidden_size'], dtype=bfloat16
   • residual: shape=['batch_size', 'hidden_size'], dtype=bfloat16
   • weight: shape=['hidden_size'], dtype=bfloat16

📤 Outputs:
   • output: shape=['batch_size', 'hidden_size'], dtype=bfloat16

📜 Reference Implementation:
------------------------------------------------------------
import torch

@torch.no_grad()
def run(hidden_states, residual, weight):
    _, hidden_size = hidden_states.shape
    # Check constants
    assert hidden_size == 4096

    EPS = 1e-5

    x = hidden_states.to(torch.float32) + residual.to(torch.float32)
    inv_rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + EP

### 2.3 Exploring Solutions (AI Agent & Human Expert Kernels)

A **Solution** is a concrete implementation of a Definition's interface. Solutions can come from:
- **AI Agents**: GPT-5, Claude Opus, Gemini 2.5 Pro, GPT-o3
- **Human Experts**: Engineers writing optimized CUDA/Triton code
- **Baseline**: FlashInfer's reference implementations


In [9]:
# List solutions for our target definition
from collections import defaultdict
solutions = trace_set.solutions.get(def_name, [])

print("="*60)
print(f"      SOLUTIONS FOR: {def_name}")
print("="*60)
print(f"\nTotal solutions: {len(solutions)}\n")

# Group by author
by_author = defaultdict(list)
for sol in solutions:
    by_author[sol.author].append(sol)

for author, sols in sorted(by_author.items()):
    print(f"\n👤 Author: {author}")
    for sol in sols:
        lang = sol.spec.language.value if hasattr(sol.spec.language, 'value') else sol.spec.language
        target = ", ".join(sol.spec.target_hardware) if sol.spec.target_hardware else "any"
        print(f"   └── {sol.name}")
        print(f"       Language: {lang} | Target: {target}")


      SOLUTIONS FOR: fused_add_rmsnorm_h4096

Total solutions: 9


👤 Author: claude-opus-4-1-20250805
   └── claude-opus-4-1_cuda_462ef5
       Language: cuda | Target: B200
   └── claude-opus-4-1_triton_f41fa3
       Language: triton | Target: B200

👤 Author: flashinfer
   └── flashinfer_wrapper_0ff432
       Language: python | Target: NVIDIA GeForce RTX 4090, NVIDIA A100, NVIDIA H20, NVIDIA H100, NVIDIA H200, NVIDIA B200

👤 Author: gemini-2.5-pro
   └── gemini-2.5-pro_cuda_5808cd
       Language: cuda | Target: B200
   └── gemini-2.5-pro_triton_dc28mj
       Language: triton | Target: B200

👤 Author: gpt-5-2025-08-07
   └── gpt-5_cuda_727b5d
       Language: cuda | Target: B200
   └── gpt-5_triton_0de5b5
       Language: triton | Target: B200

👤 Author: gpt-o3
   └── gpt-o3_cuda_a7bbcf
       Language: cuda | Target: B200
   └── gpt-o3_triton_c1e819
       Language: triton | Target: B200


In [10]:
# Look at an AI-generated solution (GPT-5 Triton)
gpt5_sol = next((s for s in solutions if "gpt-5" in s.author.lower() and "triton" in s.spec.language.value.lower()), None)

if gpt5_sol:
    print("="*60)
    print(f"      AI-GENERATED KERNEL: {gpt5_sol.name}")
    print("="*60)
    print(f"\n🤖 Author: {gpt5_sol.author}")
    print(f"📝 Description: {gpt5_sol.description}")
    print(f"🛠️  Language: {gpt5_sol.spec.language.value}")
    print(f"🎯 Target Hardware: {gpt5_sol.spec.target_hardware}")
    print(f"📂 Entry Point: {gpt5_sol.spec.entry_point}")
    
    print(f"\n📜 Generated Kernel Code (first 1500 chars):")
    print("-"*60)
    for src in gpt5_sol.sources:
        print(f"# File: {src.path}")
        print(src.content[:1500])
        if len(src.content) > 1500:
            print("\n... (truncated)")
    print("-"*60)


      AI-GENERATED KERNEL: gpt-5_triton_0de5b5

🤖 Author: gpt-5-2025-08-07
📝 Description: gpt-5-2025-08-07 optimized kernel for fused_add_rmsnorm_h4096 (round 1, reasoning effort: high)
🛠️  Language: triton
🎯 Target Hardware: ['B200']
📂 Entry Point: main.py::run

📜 Generated Kernel Code (first 1500 chars):
------------------------------------------------------------
# File: main.py
import torch
import triton
import triton.language as tl


@triton.jit
def fused_add_rmsnorm_h4096_kernel(
    hidden_ptr, residual_ptr, weight_ptr, output_ptr,
    M,  # number of rows (batch size)
    stride_hs_m, stride_hs_n,
    stride_res_m, stride_res_n,
    stride_out_m, stride_out_n,
    H: tl.constexpr,       # hidden size, must be 4096
    EPS: tl.constexpr,     # epsilon for numerical stability
    BLOCK_SIZE: tl.constexpr,
):
    tl.static_assert(H == 4096)
    pid = tl.program_id(0)
    row_in_bounds = pid < M

    cols = tl.arange(0, BLOCK_SIZE)

    # First pass: compute sum of squares across t

---
## 3. The Leaderboard

The **FlashInfer-Bench Leaderboard** ([bench.flashinfer.ai](https://bench.flashinfer.ai/)) provides:
- **Global Author Ranking**: Aggregates performance metrics to rank contributors
- **Drill-Down Analysis**: Detailed insights into specific kernel definitions
- **fast_p Metric**: Fraction of workloads where a kernel runs 'p' times faster than baseline


In [11]:
# Analyze author performance across all definitions
import pandas as pd
author_stats = defaultdict(lambda: {"total": 0, "passed": 0, "total_speedup": 0, "wins": 0})

for def_name_iter, traces_list in trace_set.traces.items():
    for trace in traces_list:
        sol = trace_set.get_solution(trace.solution)
        if sol:
            author = sol.author
            author_stats[author]["total"] += 1
            
            if trace.evaluation and trace.evaluation.status.value == "PASSED":
                author_stats[author]["passed"] += 1

print("="*80)
print("         AUTHOR LEADERBOARD (LOCAL DATASET)")
print("="*80)

# Calculate average speedup and sort
leaderboard = []
for author, stats in author_stats.items():
    if stats["passed"] > 0:
        avg_speedup = stats["total_speedup"] / stats["passed"]
        win_rate = stats["wins"] / stats["passed"] * 100
        leaderboard.append({
            "Author": author,
            "Total Traces": stats["total"],
            "Passed": stats["passed"],
        })

leaderboard_df = pd.DataFrame(leaderboard)
leaderboard_df = leaderboard_df.sort_values("Passed", ascending=False)
print(leaderboard_df.to_string(index=False))

print("\n📊 Visit https://bench.flashinfer.ai/ for the full interactive leaderboard!")

         AUTHOR LEADERBOARD (LOCAL DATASET)
                  Author  Total Traces  Passed
        gpt-5-2025-08-07          1295    1087
                  gpt-o3          1320     941
          gemini-2.5-pro          1295     632
claude-opus-4-1-20250805          1295     550
              flashinfer           392     392
                 PyTorch           311     277

📊 Visit https://bench.flashinfer.ai/ for the full interactive leaderboard!


---
## 4. End-to-End Apply with SGLang & Llama 3.1

The **Apply** feature allows you to replace kernels in FlashInfer with the best-performing ones from the trace database. This is the key to the "virtuous cycle" - improvements flow directly to production!

### How it works
When you call enable_apply(), FlashInfer-Bench automatically installs lightweight adapters that:
- Intercept FlashInfer wrapper methods (plan and run)
- Extract runtime parameters and match them to definitions
- Dispatch to the best-performing solution from your traces
- Fall back to the original FlashInfer implementation if no suitable solution exists


### 4.1 Starting SGLang serving with and without FlashInfer-Bench Apply

To enable kernel substitution, we set environment variables before launching SGLang:


In [2]:
print("="*70)
print("      Step 1: Launch SGLang serving endpoint with FlashInfer-Bench Apply")
print("="*70)
print("""
# Set environment variables to enable kernel substitution
export FIB_ENABLE_APPLY=1
export FIB_DATASET_PATH=/sgl-workspace/flashinfer-trace

# Launch SGLang server with Llama-3.1-8B-Instruct
cd /sgl-workspace/sglang
python3 -m sglang.launch_server \\
    --model-path meta-llama/Llama-3.1-8B-Instruct \\
    --cuda-graph-max-bs 8 \\
    --disable-radix-cache

# You should see "FlashInfer-Bench Apply for <kernel_name>" messages
# indicating that optimized kernels are being substituted!
""")


      Step 1: Launch SGLang serving endpoint with FlashInfer-Bench Apply

# Set environment variables to enable kernel substitution
export FIB_ENABLE_APPLY=1
export FIB_DATASET_PATH=/sgl-workspace/flashinfer-trace

# Launch SGLang server with Llama-3.1-8B-Instruct
cd /sgl-workspace/sglang
python3 -m sglang.launch_server \
    --model-path meta-llama/Llama-3.1-8B-Instruct \
    --cuda-graph-max-bs 8 \
    --disable-radix-cache

# You should see "FlashInfer-Bench Apply for <kernel_name>" messages
# indicating that optimized kernels are being substituted!



### 4.2 Benchmarking SGLang serving

Once the server is running, use `sglang.bench_serving` to measure performance:


In [3]:
print("="*70)
print("      Step 2: Collect the Serving perf numbers")
print("="*70)
print("""
# Set target model and batch size
tgt_model="meta-llama/Llama-3.1-8B-Instruct"
bs=8  # Concurrency

# Run the benchmark
python3 -m sglang.bench_serving \\
    --backend sglang \\
    --model ${tgt_model} \\
    --num-prompts $((50 * bs)) \\
    --sharegpt-output-len 100 \\
    --max-concurrency $bs

# Example with different batch sizes:
for bs in 1 2 4 8; do
    echo "=== Benchmarking with concurrency=$bs ==="
    python3 -m sglang.bench_serving \\
        --backend sglang \\
        --model meta-llama/Llama-3.1-8B-Instruct \\
        --num-prompts $((50 * bs)) \\
        --sharegpt-output-len 100 \\
        --max-concurrency $bs
done
""")


      Step 2: Collect the Serving perf numbers

# Set target model and batch size
tgt_model="meta-llama/Llama-3.1-8B-Instruct"
bs=8  # Concurrency

# Run the benchmark
python3 -m sglang.bench_serving \
    --backend sglang \
    --model ${tgt_model} \
    --num-prompts $((50 * bs)) \
    --sharegpt-output-len 100 \
    --max-concurrency $bs

# Example with different batch sizes:
for bs in 1 2 4 8; do
    echo "=== Benchmarking with concurrency=$bs ==="
    python3 -m sglang.bench_serving \
        --backend sglang \
        --model meta-llama/Llama-3.1-8B-Instruct \
        --num-prompts $((50 * bs)) \
        --sharegpt-output-len 100 \
        --max-concurrency $bs
done



---
## 5. Summary

### Key Takeaways:

1. **FlashInfer-Bench** creates a virtuous cycle where AI agents and human experts collaborate to optimize GPU kernels.

2. **The Pipeline**:
   - **Definitions**: Formal specifications of kernel interfaces
   - **Solutions**: Implementations from AI (GPT-5, Claude, Gemini) and humans
   - **Workloads**: Real-world input shapes captured from production
   - **Evaluations**: Correctness and performance measurements

3. **The Leaderboard** ([bench.flashinfer.ai](https://bench.flashinfer.ai/)) ranks contributors and tracks the best kernels.

4. **End-to-End Apply with SGLang**:
   ```bash
   # Enable kernel substitution
   export FIB_ENABLE_APPLY=1
   export FIB_DATASET_PATH=/sgl-workspace/flashinfer-trace
   
   # Launch server
   python3 -m sglang.launch_server --model-path meta-llama/Llama-3.1-8B-Instruct ...
   
   # Benchmark
   python3 -m sglang.bench_serving --backend sglang --model ... --num-prompts ...
   ```

5. **Works with SGLang, vLLM, and other LLM frameworks** - improvements flow directly to production!
